# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window

# Reading from Bronze

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

# Data transformations

In [0]:
df.display()

Analyzing the table, we could replace null values for 0's, rename the columns and change the 'S', 'M' and 'R' in the prd_line to be more understandable.

# Trimming

In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
     


# Procut Key Parsing

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

## Renaming Colums

In [0]:
RENAMING_MAP = {
    "prd_id" : "production_id",
    "cat_id": "category_id",
    "prd_key" : "production_key",
    "prd_nm" : "production_name",
    "prd_cost" : "production_cost",
    "prd_line" : "production_line",
    "prd_start_dt" : "production_start_date",
    "prd_end_dt" : "production_end_date"
}

for old_name, new_name in RENAMING_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

df.display()

## Replacing null values to 0

In [0]:
df.fillna({"production_cost": 0}).show()

## Normalizing Product Line

In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "product_line",
        F.when(F.upper(col("production_line")) == "M", "Mountain")
         .when(F.upper(col("production_line")) == "R", "Road")
         .when(F.upper(col("production_line")) == "S", "Other Sales")
         .when(F.upper(col("production_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

## Sanity check of final dataframe

In [0]:
df.limit(10).display()

## Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

## Sanity check of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10